# Cleaning and Feature Engineering

## Library import and settings

In [64]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.float_format', '{:.4f}'.format)


## Load the Dataset

In [ ]:
listings = pd.read_csv("../data/raw/listings.csv")


## Shortly reminder


In [66]:
listings.info()


<class 'pandas.DataFrame'>
RangeIndex: 2852 entries, 0 to 2851
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              2852 non-null   int64  
 1   name                            2852 non-null   str    
 2   host_id                         2852 non-null   int64  
 3   host_name                       2849 non-null   str    
 4   neighbourhood_group             0 non-null      float64
 5   neighbourhood                   2852 non-null   int64  
 6   latitude                        2852 non-null   float64
 7   longitude                       2852 non-null   float64
 8   room_type                       2852 non-null   str    
 9   price                           2537 non-null   float64
 10  minimum_nights                  2852 non-null   int64  
 11  number_of_reviews               2852 non-null   int64  
 12  last_review                     2609 non-null

## Feature Transformations


### Transform `last_review` from `str` to `date` type

In [67]:
listings["last_review"] = pd.to_datetime(listings["last_review"])


### Transform `neighbourhood` from `int64` to `category` type

In [68]:
listings['neighbourhood'] = listings['neighbourhood'].astype('category')


### Drop empty columns

In [69]:
listings.drop(columns=["neighbourhood_group", "license"], inplace=True)


## Add Derived Features

### Useful statistics

In [70]:
Q1 = listings["price"].quantile(0.25)
Q3 = listings["price"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


### Feature `has_last_review`

In [71]:
listings["has_last_review"] = listings["last_review"].notna()


### Feature `has_price`

In [72]:
listings["has_price"] = listings["price"].notna()


### Feature `is_price_outlier`

In [73]:
listings["is_price_outlier"] = listings["price"] > upper_bound


### Feature `host_size_segment`

In [74]:
bins = [1, 2, 5, 999999]
labels = ["Indivual host", "Experienced host", "Industrial host"]

listings["host_size_segment"] = pd.cut(listings["calculated_host_listings_count"], bins=bins, labels=labels, right=False)


### Feature `availability_ratio`

In [75]:
listings["availability_ratio"] = listings["availability_365"] / 365


### Feature `review_intensity`

In [76]:
listings["review_intensity"] = np.where(
    listings["number_of_reviews"] == 0,
    0,
    listings["number_of_reviews_ltm"] / listings["number_of_reviews"]
)


## Save changes

In [77]:
listings.to_csv("../data/processed/listings_analytics.csv")


##